<a href="https://colab.research.google.com/github/ruchiradevane/Java-Jenkins-Demo/blob/main/SupervisedMorfessorIndicnlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets morfessor indic-nlp-library

In [ ]:
#Download Dataset
from datasets import load_dataset

dataset = load_dataset("karthika95/morphTok")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

hindi.csv: 0.00B [00:00, ?B/s]

marathi.csv: 0.00B [00:00, ?B/s]

tamil.csv:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/279938 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['word', 'segmentation', 'language'],
        num_rows: 279938
    })
})


In [ ]:
#Inspect Data
print(dataset['train'][0])

{'word': 'रसदशा', 'segmentation': 'रस+दशा', 'language': 'hi'}


In [ ]:
#Filter Hindi Only
data = [item for item in dataset['train'] if item['language'] == 'hi']

In [ ]:
#Train-Test Split
from sklearn.model_selection import train_test_split

train_data_raw, test_data_raw = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
#Use Indic NLP (Hindi preprocessing)
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

factory = IndicNormalizerFactory()
normalizer = factory.get_normalizer("hi")


In [ ]:
#Prepare Supervised Training Data
train_data = []

for item in train_data_raw:
    word = normalizer.normalize(item['word'])

    segmented = item['segmentation'].replace("+", " ")

    train_data.append((1.0, segmented))

In [ ]:
#Step 9: Train Morfessor (Supervised Style)
import morfessor

model = morfessor.BaselineModel()

# strengthen supervision
train_data = train_data * 10

model.load_data(train_data)
model.train_batch()

100% (57456 of 57456) |##################| Elapsed Time: 0:00:12 Time:  0:00:12
100% (57456 of 57456) |##################| Elapsed Time: 0:00:12 Time:  0:00:12


(2, 3212096779.1762056)

In [ ]:
#test tokenization
test_words = ["रसदशा", "विद्यालय", "अनुशासन", "सुंदरता"]

for word in test_words:
    word = normalizer.normalize(word)

    pred, _ = model.viterbi_segment(word)
    print(word, "->", " + ".join(pred))

रसदशा -> रसद + शा
विद्यालय -> विद्या + लय
अनुशासन -> अनु + शासन
सुंदरता -> सुं + दरता


In [ ]:
correct = 0
total = 0

for item in test_data_raw:
    word = normalizer.normalize(item['word'])

    true = item['segmentation'].split("+")

    pred, _ = model.viterbi_segment(word)

    if pred == true:
        correct += 1
    total += 1

print("Accuracy:", correct / total)

Accuracy: 0.38686686963742045


In [ ]:
test_words = [
    "अनुकरण", "पुनर्निर्माण", "निराकरण",
    "सुंदरता", "मित्रता", "कठोरता",
    "राजमहल", "जलपान", "राष्ट्रभाषा",
    "स्वतंत्रता", "प्रशासन", "विकासशील"
]

for word in test_words:
    word = normalizer.normalize(word)

    pred, _ = model.viterbi_segment(word)
    print(word, "->", " + ".join(pred))

अनुकरण -> अनुकरण
पुनर्निर्माण -> पुनर् + निर्माण
निराकरण -> निराकरण
सुंदरता -> सुं + दरता
मित्रता -> मित् + रता
कठोरता -> कठोर + ता
राजमहल -> राज + महल
जलपान -> जल + पा + न
राष्ट्रभाषा -> राष्ट्र + भाषा
स्वतंत्रता -> स्वतंत्रता
प्रशासन -> प्रशासन
विकासशील -> विकासशील


In [ ]:
print("Total samples (Hindi only):", len(data))

Total samples (Hindi only): 72257


In [ ]:
dataset = dataset['train'].shuffle(seed=42)
split = dataset.train_test_split(test_size=0.2)

train_dataset = split['train']
test_dataset = split['test']

In [ ]:
print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 223950
Testing samples: 55988


In [ ]:
train_data = train_data * 3

In [ ]:
print("Actual training instances:", len(train_data))

Actual training instances: 520245


In [ ]:
print(test_data_raw[0])

{'word': 'धमकूं', 'segmentation': 'धमक+ूं', 'language': 'hi'}
